# Análisis Exploratorio de Datos (EDA)

**Seminario de Actualización — Unidad 5**
Práctica principal: *profiling* y diagnóstico de calidad de datos

## Objetivos de la clase

1. Ejecutar el recorrido de reconocimiento de un conjunto de datos nuevo: forma, tipos, muestra.
2. Detectar y cuantificar valores faltantes, duplicados, atípicos e inconsistencias de categorías.
3. Distinguir entre problemas estadísticos y violaciones de reglas de negocio.
4. Decidir dónde conviene perfilar: en pandas o en el motor.
5. Redactar un diagnóstico de calidad de datos que sostenga las decisiones de transformación.

## Sobre los datos de esta clase

El conjunto que genera `datos_demo.py` tiene **problemas de calidad deliberados**. La
consigna de la clase es encontrarlos todos sin mirar el código fuente del generador.

In [1]:
# ---------------------------------------------------------------------
# Conjunto de trabajo: la vista ValoresPedido unida a Clientes.
#
# ValoresPedido tiene IDPedido, IDCliente, IDEmpleado, EnvioPor, FechaPedido,
# FechaRequerida, FechaEnvio, cantidad y val. NO tiene Region, Pais ni Ciudad:
# esas columnas están en Clientes y hay que traerlas con un JOIN.
# ---------------------------------------------------------------------
import pandas as pd
import config_conexion
import datos_demo

CONSULTA = """
    SELECT v.IDPedido, v.IDCliente, v.IDEmpleado, v.EnvioPor,
           v.FechaPedido, v.FechaRequerida, v.FechaEnvio,
           v.cantidad, v.val,
           c.NombreEmpresa, c.Ciudad, c.Region, c.Pais
    FROM ValoresPedido AS v
      JOIN Clientes AS c
        ON c.IDCliente = v.IDCliente
"""

disponible = config_conexion.probar_conexion()

if disponible:
    engine = config_conexion.crear_engine()
    df = pd.read_sql(CONSULTA, engine)
    ORIGEN = "SQL Server"
else:
    engine = None
    tablas = datos_demo.generar_todo()
    df = tablas["ValoresPedido"].merge(
        tablas["Clientes"][["IDCliente", "NombreEmpresa", "Ciudad", "Region", "Pais"]],
        on="IDCliente", how="inner")
    ORIGEN = "datos sintéticos (datos_demo.py)"

print(f"Origen de los datos: {ORIGEN}")
print(f"Filas: {len(df)}   Columnas: {df.shape[1]}")
print(f"Columnas: {list(df.columns)}")

No se pudo conectar a SQL Server: OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 18 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [2].  (2) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 18 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to localhost. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (2)')
(Background on this error at: https://sqlalche.me/e/21/e3q8)
Origen de los datos: datos sintéticos (datos_demo.py)
Filas: 870   Columnas: 13
Columnas: ['IDPedido', 'IDCliente', 'IDEmpleado', 'EnvioPor', 'FechaPedido', 'FechaRequerida', 'FechaEnvio', 'cantidad', 'val', 'NombreEmpresa', 'Ciudad', 'Region', 'Pais']


## 1. El primer vistazo

Cuatro instrucciones que deberían ser un reflejo cada vez que se carga un `DataFrame` nuevo.

Sobre `head()` y `sample()`: si la tabla llegó ordenada por fecha, las primeras diez filas
son todas del período más antiguo y dan una impresión equivocada del conjunto. `sample()`
devuelve filas al azar y suele revelar antes la variedad real.

In [2]:
print("Forma (filas, columnas):", df.shape)
print()
print("Tipos de dato:")
print(df.dtypes)

Forma (filas, columnas): (870, 13)

Tipos de dato:
IDPedido                   int64
IDCliente                 object
IDEmpleado                 int64
EnvioPor                   int64
FechaPedido               object
FechaRequerida    datetime64[ns]
FechaEnvio        datetime64[ns]
cantidad                   int64
val                      float64
NombreEmpresa             object
Ciudad                    object
Region                    object
Pais                      object
dtype: object


In [3]:
df.head(5)

,IDPedido,IDCliente,IDEmpleado,EnvioPor,FechaPedido,FechaRequerida,FechaEnvio,cantidad,val,NombreEmpresa,Ciudad,Region,Pais
0,11027,EWHZJ,7,1,2020-07-12 15:06:00,2020-08-09,2020-08-15,26,1128.42,Empresa EWHZJ,Ciudad 36,WA,Dinamarca
1,10749,MCCJA,7,2,2020-11-01 11:17:00,2020-11-29,2020-11-19,105,6531.83,Empresa MCCJA,Ciudad 17,None,Reino Unido
2,11039,ORYAY,6,3,2019-09-30 14:46:00,2019-10-28,2019-10-31,167,1376.54,Empresa ORYAY,Ciudad 43,BC,Francia
3,10758,GFBCR,8,2,2021-01-21 18:44:00,2021-02-18,2021-01-31,73,1648.23,Empresa GFBCR,Ciudad 26,None,Portugal
4,10394,JFEIL,3,2,2019-10-08 08:18:00,2019-11-05,2019-11-02,34,1094.60,Empresa JFEIL,Ciudad 69,Centro,México


In [4]:
df.sample(5, random_state=1)

,IDPedido,IDCliente,IDEmpleado,EnvioPor,FechaPedido,FechaRequerida,FechaEnvio,cantidad,val,NombreEmpresa,Ciudad,Region,Pais
291,10455,ARPIO,3,1,2021-04-30 15:08:00,2021-05-28,2021-05-30,7,61.40,Empresa ARPIO,Ciudad 22,None,Dinamarca
495,10804,AMKXP,6,1,2020-04-30 08:09:00,2020-05-28,2020-05-30,3,46.95,Empresa AMKXP,Ciudad 11,None,Australia
180,10790,WUSQA,6,3,2019-08-16 09:17:00,2019-09-13,2019-09-02,82,4074.94,Empresa WUSQA,Ciudad 62,None,Canadá
185,10984,XZGTP,2,3,2020-08-28 15:27:00,2020-09-25,2020-09-16,52,2665.90,Empresa XZGTP,Ciudad 05,None,Francia
433,10925,VUSXS,8,3,2020-01-26 09:07:00,2020-02-23,2020-01-29,60,2386.15,Empresa VUSXS,Ciudad 55,None,Alemania


### Atención a los tipos

Revisar `dtypes` no es un trámite. El síntoma más frecuente es una columna que **debería**
ser numérica o de fecha y aparece como texto (`object` en versiones anteriores de pandas,
`str` desde pandas 3.0, que introdujo un tipo dedicado para cadenas).

En este conjunto hay al menos una columna con ese problema. Encontrala.

In [5]:
# ¿Qué columnas llegaron como texto?
columnas_texto = df.columns[df.dtypes.astype(str).isin(["object", "str", "string"])]
print("Columnas de texto:", list(columnas_texto))

# ¿Alguna de ellas debería ser otra cosa?
for col in columnas_texto:
    print(f"\n{col}: {df[col].dropna().head(3).tolist()}")

Columnas de texto: ['IDCliente', 'FechaPedido', 'NombreEmpresa', 'Ciudad', 'Region', 'Pais']

IDCliente: ['EWHZJ', 'MCCJA', 'ORYAY']

FechaPedido: ['2020-07-12 15:06:00', '2020-11-01 11:17:00', '2019-09-30 14:46:00']

NombreEmpresa: ['Empresa EWHZJ', 'Empresa MCCJA', 'Empresa ORYAY']

Ciudad: ['Ciudad 36', 'Ciudad 17', 'Ciudad 43']

Region: ['WA', 'BC', 'Centro']

Pais: ['Dinamarca', 'Reino Unido', 'Francia']


In [6]:
# Corrección del tipo de FechaPedido. errors="coerce" convierte a NaT lo que no se pueda parsear,
# en lugar de interrumpir la ejecución: conviene revisar después cuántos NaT quedaron.
df["FechaPedido"] = pd.to_datetime(df["FechaPedido"], errors="coerce")
df["val"] = pd.to_numeric(df["val"], errors="coerce")

print(df.dtypes)
print("\nFechas no parseables:", df["FechaPedido"].isna().sum())

IDPedido                   int64
IDCliente                 object
IDEmpleado                 int64
EnvioPor                   int64
FechaPedido       datetime64[ns]
FechaRequerida    datetime64[ns]
FechaEnvio        datetime64[ns]
cantidad                   int64
val                      float64
NombreEmpresa             object
Ciudad                    object
Region                    object
Pais                      object
dtype: object

Fechas no parseables: 0


## 2. `info()` y `describe()`

`info()` reúne cantidad de filas, tipo y no nulos por columna, y uso de memoria: la forma
más rápida de ver dónde hay faltantes.

`describe()` agrega los estadísticos descriptivos. La lectura interesante está en los
contrastes: media muy lejos de la mediana (distribución asimétrica), mínimo negativo donde
no debería haberlo, máximo desproporcionado respecto del percentil 99, desvío cero
(columna constante).

In [7]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 870 entries, 0 to 869
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   IDPedido        870 non-null    int64         
 1   IDCliente       870 non-null    object        
 2   IDEmpleado      870 non-null    int64         
 3   EnvioPor        870 non-null    int64         
 4   FechaPedido     870 non-null    datetime64[ns]
 5   FechaRequerida  870 non-null    datetime64[ns]
 6   FechaEnvio      848 non-null    datetime64[ns]
 7   cantidad        870 non-null    int64         
 8   val             870 non-null    float64       
 9   NombreEmpresa   870 non-null    object        
 10  Ciudad          870 non-null    object        
 11  Region          275 non-null    object        
 12  Pais            870 non-null    object        
dtypes: datetime64[ns](3), float64(1), int64(4), object(5)
memory usage: 311.7 KB


In [8]:
df.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2)

,IDPedido,IDEmpleado,EnvioPor,FechaPedido,FechaRequerida,FechaEnvio,cantidad,val
count,870.00,870.00,870.00,870,870,848,870.00,870.00
mean,10661.62,5.04,1.98,2020-06-07 02:26:33.241379328,2020-07-04 12:52:57.931034368,2020-06-24 05:20:56.603773440,68.83,1740.25
min,10248.00,1.00,1.00,2019-07-04 08:45:00,2019-08-01 00:00:00,2019-07-06 00:00:00,2.00,30.91
1%,10255.69,1.00,1.00,2019-07-09 03:37:55.200000,2019-08-05 16:33:36,2019-07-21 00:00:00,5.00,49.65
25%,10453.25,3.00,1.00,2020-01-03 05:28:30,2020-01-30 12:00:00,2020-01-19 18:00:00,34.00,609.94
50%,10660.50,5.00,2.00,2020-06-05 03:17:00,2020-07-02 12:00:00,2020-06-19 12:00:00,59.00,1232.72
75%,10869.75,7.00,3.00,2020-11-11 04:43:15,2020-12-08 12:00:00,2020-11-24 12:00:00,91.00,2300.82
95%,11035.55,9.00,3.00,2021-03-30 10:15:48,2021-04-27 00:00:00,2021-04-20 00:00:00,162.00,4888.10
99%,11068.31,9.00,3.00,2021-04-25 07:28:36,2021-05-22 22:19:12,2021-05-19 12:43:12,203.62,8528.47
max,11077.00,9.00,3.00,2021-05-06 18:00:00,2021-06-03 00:00:00,2021-06-03 00:00:00,328.00,13858.48


In [9]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
IDPedido,870.0,NaN,NaN,NaN,10661.62069,10248.0,10453.25,10660.5,10869.75,11077.0,239.973433
IDCliente,870,89,QYNXK,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
IDEmpleado,870.0,NaN,NaN,NaN,5.043678,1.0,3.0,5.0,7.0,9.0,2.65949
EnvioPor,870.0,NaN,NaN,NaN,1.982759,1.0,1.0,2.0,3.0,3.0,0.813254
FechaPedido,870,NaN,NaN,NaN,2020-06-07 02:26:33.241379328,2019-07-04 08:45:00,2020-01-03 05:28:30,2020-06-05 03:17:00,2020-11-11 04:43:15,2021-05-06 18:00:00,NaN
FechaRequerida,870,NaN,NaN,NaN,2020-07-04 12:52:57.931034368,2019-08-01 00:00:00,2020-01-30 12:00:00,2020-07-02 12:00:00,2020-12-08 12:00:00,2021-06-03 00:00:00,NaN
FechaEnvio,848,NaN,NaN,NaN,2020-06-24 05:20:56.603773440,2019-07-06 00:00:00,2020-01-19 18:00:00,2020-06-19 12:00:00,2020-11-24 12:00:00,2021-06-03 00:00:00,NaN
cantidad,870.0,NaN,NaN,NaN,68.829885,2.0,34.0,59.0,91.0,328.0,46.339901
val,870.0,NaN,NaN,NaN,1740.251563,30.91,609.94,1232.715,2300.8175,13858.48,1719.689677
NombreEmpresa,870,89,Empresa QYNXK,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Lectura guiada de describe(): tres señales concretas
monto = df["val"]
print(f"Media   : {monto.mean():>12,.2f}")
print(f"Mediana : {monto.median():>12,.2f}   <- muy por debajo de la media: distribución asimétrica")
print(f"Máximo  : {monto.max():>12,.2f}")
print(f"P99     : {monto.quantile(0.99):>12,.2f}   <- el máximo está muy por encima del P99")
print(f"Mínimo  : {monto.min():>12,.2f}   <- ¿un monto negativo tiene sentido de negocio?")
print()
print("Columnas constantes (desvío 0 o un único valor):")
for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        print(f"   - {col}: siempre '{df[col].iloc[0]}'")

Media   :     1,740.25
Mediana :     1,232.72   <- muy por debajo de la media: distribución asimétrica
Máximo  :    13,858.48
P99     :     8,528.47   <- el máximo está muy por encima del P99
Mínimo  :        30.91   <- ¿un monto negativo tiene sentido de negocio?

Columnas constantes (desvío 0 o un único valor):


## 3. Valores faltantes

`NaN` (numérico), `NaT` (fecha) y `pd.NA` provienen habitualmente de un `NULL` de SQL Server.
Vale la lógica de tres valores vista al comienzo del seminario: `NULL` no es cero ni cadena
vacía, y `NaN != NaN` es verdadero. Por eso se detectan con `isna()`, nunca con `==`.

In [11]:
faltantes = pd.DataFrame({
    "Nulos": df.isna().sum(),
    "Porcentaje": (df.isna().mean() * 100).round(2),
})
faltantes[faltantes["Nulos"] > 0].sort_values("Nulos", ascending=False)

,Nulos,Porcentaje
Region,595,68.39
FechaEnvio,22,2.53


In [12]:
# El riesgo silencioso: las agregaciones ignoran los faltantes sin avisar.
# Los pedidos todavía no despachados no tienen FechaEnvio, así que no tienen
# demora conocida: tienen demora MÍNIMA. Es un dato censurado, no un dato ausente.
df["FechaPedido"] = pd.to_datetime(df["FechaPedido"], errors="coerce")
df["FechaEnvio"] = pd.to_datetime(df["FechaEnvio"], errors="coerce")
df["DiasEnvio"] = (df["FechaEnvio"] - df["FechaPedido"]).dt.days

sin_despachar = df["FechaEnvio"].isna().sum()
print(f"Demora promedio sobre los despachados : {df['DiasEnvio'].mean():.1f} días")
print(f"Pedidos sin despachar                 : {sin_despachar} "
      f"({df['FechaEnvio'].isna().mean():.1%})")
print()
print("El promedio se calculó ignorando los no despachados, que son justamente")
print("los que más están tardando. Reportarlo sin aclararlo subestima el problema.")

Demora promedio sobre los despachados : 16.8 días
Pedidos sin despachar                 : 22 (2.5%)

El promedio se calculó ignorando los no despachados, que son justamente
los que más están tardando. Reportarlo sin aclararlo subestima el problema.


In [13]:
# Estrategias, cada una con su justificación.

# Region: dos tercios de los clientes no la tienen cargada. No es un error de
# extracción: en el modelo de Pampero la región solo aplica a algunos países.
# Marcarla permite agrupar después sin perder esas filas en silencio.
df["Region"] = df["Region"].fillna("Sin región")

# IDCliente y val son obligatorios para cualquier análisis de ventas:
# si faltan, la fila no sirve.
antes = len(df)
df = df.dropna(subset=["IDCliente", "val"])
print(f"Filas descartadas por faltantes críticos: {antes - len(df)}")
print(f"Clientes marcados como 'Sin región': {(df['Region'] == 'Sin región').sum()}")

Filas descartadas por faltantes críticos: 0
Clientes marcados como 'Sin región': 595


## 4. Duplicados

Un `DataFrame` no tiene clave primaria: nada impide dos filas idénticas. La causa habitual
es una extracción ejecutada dos veces, o un `JOIN` mal planteado en el origen que multiplicó filas.

Antes de eliminar, **mirar**: si dos filas comparten el identificador pero difieren en el
monto, no es una carga repetida sino una inconsistencia real.

In [14]:
print("Filas completamente repetidas :", df.duplicated().sum())
print("Repetidas según IDPedido       :", df.duplicated(subset=["IDPedido"]).sum())
print("¿IDPedido sirve como clave?    :", df["IDPedido"].nunique() == len(df))

Filas completamente repetidas : 40
Repetidas según IDPedido       : 40
¿IDPedido sirve como clave?    : False


In [15]:
# Inspección antes de eliminar: keep=False marca todas las apariciones
repetidos = df[df.duplicated(subset=["IDPedido"], keep=False)].sort_values("IDPedido")
repetidos.head(6)

,IDPedido,IDCliente,IDEmpleado,EnvioPor,FechaPedido,FechaRequerida,FechaEnvio,cantidad,val,NombreEmpresa,Ciudad,Region,Pais,DiasEnvio
48,10254,XZGTP,9,1,2019-09-30 17:25:00,2019-10-28,2019-10-04,76,4204.58,Empresa XZGTP,Ciudad 05,Sin región,Francia,3.0
776,10254,XZGTP,9,1,2019-09-30 17:25:00,2019-10-28,2019-10-04,76,4204.58,Empresa XZGTP,Ciudad 05,Sin región,Francia,3.0
559,10299,OEZOA,1,3,2021-04-07 18:48:00,2021-05-05,2021-05-04,27,1369.77,Empresa OEZOA,Ciudad 01,Sin región,Brasil,26.0
82,10299,OEZOA,1,3,2021-04-07 18:48:00,2021-05-05,2021-05-04,27,1369.77,Empresa OEZOA,Ciudad 01,Sin región,Brasil,26.0
95,10305,UNYYX,1,2,2020-08-26 18:28:00,2020-09-23,2020-09-10,121,5627.34,Empresa UNYYX,Ciudad 17,Centro,Italia,14.0
705,10305,UNYYX,1,2,2020-08-26 18:28:00,2020-09-23,2020-09-10,121,5627.34,Empresa UNYYX,Ciudad 17,Centro,Italia,14.0


In [16]:
# ¿Son copias idénticas o registros distintos con el mismo identificador?
inconsistentes = (
    repetidos.groupby("IDPedido")["val"]
             .nunique()
             .loc[lambda s: s > 1]
)
print(f"IDPedido repetidos con montos distintos: {len(inconsistentes)}")
print("(si es 0, son cargas repetidas y se pueden eliminar con seguridad)")

antes = len(df)
df = df.drop_duplicates(subset=["IDPedido"], keep="first")
print(f"\nFilas eliminadas: {antes - len(df)}   Filas restantes: {len(df)}")

IDPedido repetidos con montos distintos: 0
(si es 0, son cargas repetidas y se pueden eliminar con seguridad)

Filas eliminadas: 40   Filas restantes: 830


## 5. Valores atípicos y reglas de negocio

Dos controles distintos que conviene no confundir:

- **Criterio estadístico (regla del IQR, Tukey):** señala valores que se apartan del resto.
  Solo marca *candidatos*: el cliente mayorista que compra cien veces más que el promedio no
  es un error, es el cliente más importante.
- **Reglas de negocio:** marcan lo que es *imposible por definición* (monto negativo, fecha
  futura, porcentaje mayor a 100). Es un control más productivo y no necesita estadística.

In [17]:
q1 = df["val"].quantile(0.25)
q3 = df["val"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

atipicos = df[(df["val"] < limite_inferior) | (df["val"] > limite_superior)]

print(f"Q1  = {q1:,.2f}")
print(f"Q3  = {q3:,.2f}")
print(f"IQR = {iqr:,.2f}")
print(f"Rango considerado normal: [{limite_inferior:,.2f} ; {limite_superior:,.2f}]")
print(f"\nAtípicos: {len(atipicos)} de {len(df)} filas ({len(atipicos) / len(df) * 100:.1f}%)")

Q1  = 597.97
Q3  = 2,269.86
IQR = 1,671.89
Rango considerado normal: [-1,909.87 ; 4,777.69]

Atípicos: 44 de 830 filas (5.3%)


In [18]:
# Los diez montos más altos: ¿casos reales o errores de carga?
df.nlargest(10, "val")[["IDPedido", "FechaPedido", "Pais", "cantidad", "val"]]

,IDPedido,FechaPedido,Pais,cantidad,val
174,10508,2020-07-10 09:05:00,España,228,13858.48
336,10882,2020-05-30 10:15:00,Polonia,158,11841.77
850,10871,2020-02-01 11:15:00,Noruega,132,11413.55
79,10248,2020-01-08 09:18:00,Suiza,143,10235.03
610,10294,2021-03-17 09:47:00,Polonia,166,9407.44
499,10458,2020-04-12 13:07:00,Canadá,140,9129.09
25,10668,2020-03-31 16:35:00,Italia,118,8907.35
443,10623,2019-08-12 14:55:00,Francia,177,8870.63
724,10296,2020-06-01 16:42:00,Dinamarca,138,8822.47
189,10850,2020-07-02 09:36:00,España,128,8396.38


In [19]:
# Validación contra reglas de negocio explícitas
reglas = {
    "Importe menor o igual a cero": df["val"] <= 0,
    "Fecha de pedido posterior a hoy": df["FechaPedido"] > pd.Timestamp.today(),
    "Cantidad menor o igual a cero": df["cantidad"] <= 0,
    "Envío anterior al pedido": df["FechaEnvio"] < df["FechaPedido"],
    "Envío fuera de término": df["FechaEnvio"] > df["FechaRequerida"],
}

for descripcion, condicion in reglas.items():
    print(f"{descripcion:35} : {condicion.sum():>4} filas")

Importe menor o igual a cero        :    0 filas
Fecha de pedido posterior a hoy     :    0 filas
Cantidad menor o igual a cero       :    0 filas
Envío anterior al pedido            :    0 filas
Envío fuera de término              :  166 filas


## 6. Variables categóricas

`value_counts()` es el equivalente exacto de un `GROUP BY ... COUNT(*)` ordenado de forma
descendente. Revela dos problemas típicos: **inconsistencia de representación** (la misma
categoría escrita de varias formas) y **cardinalidad inesperada** (un campo de texto libre
donde debería haber una lista controlada).

In [24]:
print("Valores distintos de Pais:", df["Pais"].nunique())
print()
print(df["Pais"].value_counts(dropna=False).head(20))

Valores distintos de Pais: 24

Pais
Polonia         87
Canadá          82
Portugal        70
Austria         57
España          54
Francia         51
Italia          43
Noruega         40
Finlandia       39
Australia       39
Dinamarca       36
Brasil          31
México          29
Suecia          25
Argentina       24
Suiza           23
Reino  Unido    22
Alemania        17
 EE.UU.         15
EEUU            14
Name: count, dtype: int64


Aparecen más países de los que el negocio tiene en realidad: el mismo país está
escrito de varias formas. Es el problema de consistencia del checklist, y se
corrige normalizando antes de agrupar.

Un detalle de esta base en particular: su intercalación es
`Modern_Spanish_100_CI_AI_SC_UTF8`, que **no distingue mayúsculas ni acentos**.
Eso significa que en SQL Server `'EE.UU.'` y `'ee.uu.'` se comparan como iguales,
pero al traerlos a pandas pasan a ser dos valores distintos. La limpieza que el
motor resolvía sola deja de estar cuando los datos salen de él.

In [26]:
# Normalización: recortar espacios, unificar mayúsculas y mapear equivalencias.
df["Pais"] = df["Pais"].str.strip().str.upper()

equivalencias = {
    "EEUU": "EE.UU.",
    "ESTADOS UNIDOS": "EE.UU.",
    "EE.UU.": "EE.UU.",
    "REINO  UNIDO": "REINO UNIDO",
}
df["Pais"] = df["Pais"].replace(equivalencias)

print("Valores distintos después de normalizar:", df["Pais"].nunique())
print()
print((df["Pais"].value_counts(normalize=True) * 100).round(1).head(15))

Valores distintos después de normalizar: 21

Pais
POLONIA        10.5
CANADÁ          9.9
PORTUGAL        8.4
AUSTRIA         6.9
ESPAÑA          6.5
FRANCIA         6.1
ITALIA          5.2
NORUEGA         4.8
AUSTRALIA       4.7
FINLANDIA       4.7
DINAMARCA       4.3
EE.UU.          4.2
REINO UNIDO     3.9
BRASIL          3.7
MÉXICO          3.5
Name: proportion, dtype: float64


## 7. Relaciones entre variables

Dos advertencias sobre la correlación:

1. **Correlación no implica causalidad:** ambas variables pueden depender de un tercer factor.
2. El coeficiente de Pearson que calcula `corr()` mide **solo relaciones lineales**. Dos
   variables con una relación fuerte pero curva pueden arrojar una correlación cercana a cero.
   Es la misma lección del cuarteto de Anscombe de la clase siguiente.

In [27]:
df[["cantidad", "val", "DiasEnvio"]].corr().round(3)

,cantidad,val,DiasEnvio
cantidad,1.000,0.682,0.021
val,0.682,1.000,0.001
DiasEnvio,0.021,0.001,1.000


In [28]:
df.groupby("Pais")["val"].agg(
    ["count", "mean", "median", "std", "sum"]
).round(2).sort_values("sum", ascending=False)

,count,mean,median,std,sum
Pais,,,,,
CANADÁ,82,1937.95,1672.42,1587.23,158911.91
POLONIA,87,1801.44,1284.12,1865.82,156725.40
AUSTRIA,57,1829.77,1229.25,1616.63,104297.08
PORTUGAL,70,1489.59,1057.09,1264.06,104271.60
FRANCIA,51,1914.21,1142.10,2042.01,97624.75
ESPAÑA,54,1796.72,1179.17,2278.49,97023.00
ITALIA,43,1836.64,1172.64,1776.58,78975.52
NORUEGA,40,1790.64,1301.56,2040.43,71625.52
AUSTRALIA,39,1784.71,920.88,1671.12,69603.59


In [29]:
# Tabla de contingencia entre dos categóricas
df["Trimestre"] = df["FechaPedido"].dt.quarter
pd.crosstab(df["Pais"], df["Trimestre"], normalize="index").round(3) * 100

Trimestre,1,2,3,4
Pais,,,,
ALEMANIA,41.2,17.6,23.5,17.6
ARGENTINA,29.2,8.3,37.5,25.0
AUSTRALIA,46.2,20.5,15.4,17.9
AUSTRIA,38.6,10.5,31.6,19.3
BRASIL,35.5,16.1,19.4,29.0
BÉLGICA,25.0,0.0,50.0,25.0
CANADÁ,31.7,15.9,29.3,23.2
DINAMARCA,27.8,19.4,22.2,30.6
EE.UU.,42.9,20.0,17.1,20.0


## 8. Perfilar del lado del motor

Cuando la tabla de origen tiene decenas de millones de filas, traerla completa a un
`DataFrame` solo para contar nulos es un desperdicio y muchas veces imposible por memoria.
Buena parte del perfilado se expresa en T-SQL y devuelve una sola fila.

Nótese el contraste entre `COUNT(*)` (cuenta filas) y `COUNT(columna)` (cuenta valores no
nulos): su diferencia es exactamente `df["columna"].isna().sum()`.

In [32]:
PERFILADO_SQL = """
SELECT
    COUNT(*)                              AS Filas,
    COUNT(v.IDCliente)                    AS ConCliente,
    COUNT(*) - COUNT(v.IDCliente)         AS SinCliente,
    COUNT(DISTINCT v.IDCliente)           AS ClientesDistintos,
    COUNT(v.FechaEnvio)                   AS Despachados,
    COUNT(*) - COUNT(v.FechaEnvio)        AS SinDespachar,
    MIN(v.val)                            AS ImporteMinimo,
    MAX(v.val)                            AS ImporteMaximo,
    AVG(CAST(v.val AS DECIMAL(18,2)))     AS ImportePromedio,
    MIN(v.FechaPedido)                    AS FechaDesde,
    MAX(v.FechaPedido)                    AS FechaHasta
FROM ValoresPedido AS v;
"""

DUPLICADOS_SQL = """
-- La vista agrupa por IDPedido, así que no puede devolver repetidos.
-- El control tiene sentido sobre la tabla base:
SELECT IDPedido, IDProducto, COUNT(*) AS Repeticiones
FROM [Detalles Pedido]
GROUP BY IDPedido, IDProducto
HAVING COUNT(*) > 1;
"""

disponible = config_conexion.probar_conexion()
if disponible:
    engine = config_conexion.crear_engine()
    df = pd.read_sql(CONSULTA, engine)
    ORIGEN = "SQL Server"

if disponible:
    print(pd.read_sql(PERFILADO_SQL, engine).T)
    print()
    print(pd.read_sql(DUPLICADOS_SQL, engine).head())
else:
    print("Sin servidor. Estas consultas deben ejecutarse contra la base del laboratorio:")
    print(PERFILADO_SQL)
    print(DUPLICADOS_SQL)

                                     0
Filas                              830
ConCliente                         830
SinCliente                           0
ClientesDistintos                   89
Despachados                        809
SinDespachar                        21
ImporteMinimo                     12.5
ImporteMaximo                  16387.5
ImportePromedio            1525.052072
FechaDesde         2019-07-04 16:58:00
FechaHasta         2021-05-06 18:40:00

Empty DataFrame
Columns: [IDPedido, IDProducto, Repeticiones]
Index: []


## 9. Diagnóstico de calidad

Checklist mínimo que conviene ejecutar sobre cualquier conjunto de datos y **dejar
registrado por escrito** junto con el análisis:

1. ¿Cuántas filas y columnas hay? ¿Coincide con lo esperado del origen?
2. ¿Los tipos de dato son correctos? ¿Las fechas son fechas y los números, números?
3. ¿Cuántos faltantes hay, en qué columnas, y qué criterio se adoptó para cada una?
4. ¿Hay duplicados según la clave de negocio? ¿Son repeticiones o inconsistencias?
5. ¿Los valores respetan los rangos válidos de las reglas de negocio?
6. ¿Las variables categóricas están normalizadas?
7. ¿Hay columnas constantes, que no aportan información?
8. ¿El período cubierto alcanza para responder la pregunta planteada?

In [33]:
def diagnostico(datos: pd.DataFrame, clave: str) -> pd.DataFrame:
    """Resumen de perfilado columna por columna, reutilizable en cualquier DataFrame."""
    return pd.DataFrame({
        "Tipo": datos.dtypes.astype(str),
        "NoNulos": datos.notna().sum(),
        "Nulos": datos.isna().sum(),
        "PctNulos": (datos.isna().mean() * 100).round(2),
        "Distintos": datos.nunique(dropna=True),
        "Ejemplo": [datos[c].dropna().iloc[0] if datos[c].notna().any() else None
                    for c in datos.columns],
    })


resumen_calidad = diagnostico(df, clave="IDPedido")
resumen_calidad

,Tipo,NoNulos,Nulos,PctNulos,Distintos,Ejemplo
IDPedido,int64,830,0,0.00,830,10248
IDCliente,object,830,0,0.00,89,VINET
IDEmpleado,int64,830,0,0.00,9,5
EnvioPor,int64,830,0,0.00,3,3
FechaPedido,datetime64[ns],830,0,0.00,829,2019-07-04 16:58:00
FechaRequerida,datetime64[ns],830,0,0.00,454,2019-08-01 00:00:00
FechaEnvio,datetime64[ns],809,21,2.53,387,2019-07-16 00:00:00
cantidad,int64,830,0,0.00,177,27
val,float64,830,0,0.00,795,440.0
NombreEmpresa,object,830,0,0.00,89,Vins et alcools Chevalier


In [34]:
resumen_calidad.to_csv("diagnostico_calidad.csv", encoding="utf-8")
print("Diagnóstico guardado en diagnostico_calidad.csv")
print(f"\nFilas finales tras la limpieza: {len(df)}")
print(f"Período cubierto: {df['FechaPedido'].min():%d/%m/%Y} a {df['FechaPedido'].max():%d/%m/%Y}")

Diagnóstico guardado en diagnostico_calidad.csv

Filas finales tras la limpieza: 830
Período cubierto: 04/07/2019 a 06/05/2021
